In [0]:
%python
# Databricks notebook source
from pyspark.sql import functions as F

def market_rag_with_gating(user_query, spark_session):
    """
    Simula um fluxo RAG com Gating (Jev-like) consultando a camada Gold.
    """
    
    # 1. Recuperação (Retrieval): Busca semântica/textual na tabela Gold
    # Em produção com Databricks, usar-se-ia o Databricks Vector Search. Aqui simulamos com SQL.
    query_lower = user_query.lower()
    
    context_df = spark_session.sql(f"""
        SELECT adm0_name AS country, mkt_name AS market, cm_name AS commodity, 
               year, month, avg_price, cur_name AS currency
        FROM market_prices_db.gold.agg_market_prices_summary 
        WHERE lower(cm_name) LIKE '%{query_lower}%' 
           OR lower(adm0_name) LIKE '%{query_lower}%'
        ORDER BY year DESC, month DESC
        LIMIT 5
    """)
    
    results = context_df.collect()
    
    # 2. Mecanismo de Gating (Abstenção): Se o contexto não contiver a resposta, o sistema abstém-se.
    if not results or len(results) == 0:
        return "Abstain: A informação solicitada não está presente na base de dados de preços de alimentos (Camada Gold)."
    
    # 3. Formatação do Contexto Validado (Rerank & Format)
    context_text = "Contexto de Mercado Recuperado:\n"
    for r in results:
        preco_formatado = round(r['avg_price'], 2) if r['avg_price'] else "N/A"
        context_text += (f"- Em {r['month']}/{r['year']}, no mercado {r['market']} "
                         f"({r['country']}), o produto {r['commodity']} custava em média "
                         f"{preco_formatado} {r['currency']}.\n")
        
    # 4. Construção do Prompt Final para o LLM
    final_prompt = f"""
    É um analista especialista em segurança alimentar e economia do World Food Programme (WFP).
    Responda à pergunta do utilizador utilizando ESTRITAMENTE o contexto fornecido abaixo. 
    Se a informação não estiver no contexto, não invente; diga que os dados não estão disponíveis.
    
    {context_text}
    
    Pergunta do utilizador: "{user_query}"
    Resposta:
    """
    
    return final_prompt

# ==========================================
# Testes do Motor RAG + Gating
# ==========================================

print("--- TESTE 1: Consulta Válida (Produto existente) ---")
pergunta_valida = "Maize"
resultado_valido = market_rag_with_gating(pergunta_valida, spark)
print(resultado_valido)

print("\n--- TESTE 2: Consulta Inválida (Aciona o Gating) ---")
pergunta_invalida = "Iphone 15 Pro Max"
resultado_invalido = market_rag_with_gating(pergunta_invalida, spark)
print(resultado_invalido)